# Guava Image Translation with a Conditional GAN

This notebook uses the accompanying `dataset/` folder of paired wrapped/unwrapped guava images. The images were collected and preprocessed by the project author.

Run from the project root (the folder containing `dataset/`). In Google Colab, first upload/extract the repository or clone it and `%cd` into that folder. GPU runtime recommended.

The original notebook included Google Drive mounting and preprocessing of raw photographs (EXIF rotation correction and removal of the top 15% of each image). Those cells are excluded here because the repository already contains the preprocessed images.


**Output guide:** Execute the cells in order to train this model. The training cell reports every 10 epochs and displays image comparisons; the evaluation cell prints SSIM and PSNR after training. A historical evaluation summary is preserved below so results remain visible on GitHub even without running training. Saved private Google Drive outputs are intentionally excluded.


In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import os
print("TensorFlow version:", tf.__version__)
print("Python working directory:", os.getcwd())


TensorFlow version: 2.20.0
Python working directory: /Users/burcudogan/Desktop/Guava-GAN-Image-Translation


In [3]:
from pathlib import Path

DATASET_PATH = str(Path("dataset").resolve())
if not Path(DATASET_PATH, "train", "wrapped").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root containing dataset/")

IMG_SIZE = 256      # use 512 if you get more data
BATCH_SIZE = 8
EPOCHS = 200
LAMBDA = 30         # lower = less blur


In [4]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)

    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = (img * 2.0) - 1.0

    return img


In [5]:
def make_dataset(split):
    w_dir = f"{DATASET_PATH}/{split}/wrapped"
    u_dir = f"{DATASET_PATH}/{split}/unwrapped"

    w_names = sorted(f for f in os.listdir(w_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    u_names = sorted(f for f in os.listdir(u_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    if w_names != u_names:
        raise ValueError(f"Unmatched wrapped/unwrapped filenames in {split}")
    w_files = [os.path.join(w_dir, f) for f in w_names]
    u_files = [os.path.join(u_dir, f) for f in w_names]

    ds = tf.data.Dataset.from_tensor_slices((w_files, u_files))
    ds = ds.map(lambda w, u: (load_image(w), load_image(u)))
    ds = ds.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


In [6]:
train_ds = make_dataset("train")
val_ds = make_dataset("val")


In [7]:
print("Dataset loaded successfully")
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


Dataset loaded successfully
Training batches: 2
Validation batches: 2


In [8]:
def down(f):
    return tf.keras.Sequential([
        layers.Conv2D(f, 4, 2, 'same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU()
    ])

def up(f):
    return tf.keras.Sequential([
        layers.Conv2DTranspose(f, 4, 2, 'same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU()
    ])


In [9]:
def Generator():
    inp = layers.Input([IMG_SIZE, IMG_SIZE, 3])

    d1 = down(64)(inp)
    d2 = down(128)(d1)
    d3 = down(256)(d2)
    d4 = down(512)(d3)

    u1 = up(256)(d4); u1 = layers.Concatenate()([u1, d3])
    u2 = up(128)(u1); u2 = layers.Concatenate()([u2, d2])
    u3 = up(64)(u2);  u3 = layers.Concatenate()([u3, d1])

    out = layers.Conv2DTranspose(3, 4, 2, 'same', activation='tanh')(u3)
    return tf.keras.Model(inp, out)


In [10]:
generator = Generator()


In [11]:
def Discriminator():
    inp = layers.Input([IMG_SIZE, IMG_SIZE, 3])
    tar = layers.Input([IMG_SIZE, IMG_SIZE, 3])

    x = layers.Concatenate()([inp, tar])
    x = down(64)(x)
    x = down(128)(x)
    x = down(256)(x)
    x = layers.Conv2D(1, 4, padding='same')(x)

    return tf.keras.Model([inp, tar], x)


In [12]:
disc = Discriminator()


In [13]:
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def gen_loss(fake, gen_out, real):
    adv = loss_fn(tf.ones_like(fake), fake)
    l1 = tf.reduce_mean(tf.abs(real - gen_out))
    return adv + LAMBDA * l1

def disc_loss(real, fake):
    return loss_fn(tf.ones_like(real), real) + loss_fn(tf.zeros_like(fake), fake)


In [14]:
g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


In [15]:
@tf.function
def train_step(x, y):
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = generator(x, training=True)
        dr = disc([x, y], training=True)
        df = disc([x, fake], training=True)

        gl = gen_loss(df, fake, y)
        dl = disc_loss(dr, df)

    g_opt.apply_gradients(zip(gt.gradient(gl, generator.trainable_variables),
                              generator.trainable_variables))
    d_opt.apply_gradients(zip(dt.gradient(dl, disc.trainable_variables),
                              disc.trainable_variables))


In [16]:
def show(model, ds):
    x, y = next(iter(ds))
    p = model(x, training=False)

    plt.figure(figsize=(12,4))
    for i in range(min(3, int(x.shape[0]))):
        plt.subplot(3,3,i+1); plt.imshow((x[i]+1)/2); plt.axis("off")
        plt.subplot(3,3,i+4); plt.imshow((p[i]+1)/2); plt.axis("off")
        plt.subplot(3,3,i+7); plt.imshow((y[i]+1)/2); plt.axis("off")
    plt.show()


In [ ]:
for e in range(EPOCHS):
    for x, y in train_ds:
        train_step(x, y)

    if (e+1) % 10 == 0:
        print("Epoch", e+1)
        show(generator, val_ds)


2026-10-08 22:43:08.994218: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-08 22:43:14.717252: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-08 22:43:25.795481: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-08 22:43:50.438480: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
import tensorflow as tf
import numpy as np

def evaluate_model(generator, val_ds):
    ssim_scores = []
    psnr_scores = []

    for x, y in val_ds:
        preds = generator(x, training=False)

        # convert from [-1,1] → [0,1]
        preds = (preds + 1) / 2
        y = (y + 1) / 2

        ssim = tf.image.ssim(preds, y, max_val=1.0)
        psnr = tf.image.psnr(preds, y, max_val=1.0)

        ssim_scores.extend(ssim.numpy())
        psnr_scores.extend(psnr.numpy())

    print("Quantitative Evaluation")
    print(f"SSIM  : {np.mean(ssim_scores):.3f}")
    print(f"PSNR  : {np.mean(psnr_scores):.2f} dB")

    return np.mean(ssim_scores), np.mean(psnr_scores)


## Results recorded in the original notebook (historical)

These numbers were printed by the original notebook at the time of that experiment. They are **not** generated by opening this cleaned notebook and are **not** guaranteed to match a new training run. The original notebook did not contain exported model weights, so you must retrain to calculate new evaluation results.

- **SSIM:** 0.468
- **PSNR:** 14.36 dB
- **Structural similarity:** approximately 46.8% (SSIM × 100; not a classification accuracy score)

The training cell above prints the epoch number and shows wrapped inputs, predicted images, and targets every 10 epochs. Run training and then run the evaluation cells below to print **new** SSIM and PSNR values.


In [ ]:
ssim_score, psnr_score = evaluate_model(generator, val_ds)


In [ ]:
print(f"Structural similarity ≈ {ssim_score*100:.1f}%")
